In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

drive_folder_name = "ProductVoiceAnalytics"
os.chdir(f'/content/drive/MyDrive/Projects/GitHub/{drive_folder_name}')

!pwd

In [ ]:
from collections import Counter
import gzip
import json
from src.config import RAW_REVIEWS_PATH, RAW_METADATA_PATH

# count reviews per ASIN
print('Counting reviews per ASIN...')
asin_counts = Counter()

with gzip.open(RAW_REVIEWS_PATH, 'rt', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        asin   = record.get('asin')
        if asin:
            asin_counts[asin] += 1

# get top 20 ASINs by review count
top_asins = [asin for asin, _ in asin_counts.most_common(20)]
print(f'Top ASINs found: {top_asins}')

# look up product names from metadata
print('Looking up product names...')
asin_names = {}

with gzip.open(RAW_METADATA_PATH, 'rt', encoding='utf-8') as f:
    for line in f:
        try:
            record = json.loads(line)
            asin   = record.get('asin')
            title  = record.get('title', '').strip()
            if asin in top_asins and title:
                asin_names[asin] = title
        except:
            continue

# build demo_products dict — top 10 that have a title
demo_products = {asin: asin_names[asin] for asin in top_asins if asin in asin_names}
demo_products_ = dict(list(demo_products.items())[:10])

print(demo_products_.items())


In [ ]:
print('\ndemo_products = {')
for asin, name in demo_products_.items():
    print(f"    '{asin}': '{name}',")
print('}')

In [ ]:
!pip install -r requirements.txt

In [ ]:
import json
from src.utils import stream_reviews_for_asin
from src.pipeline.preprocess import clean_text
from src.pipeline.sentiment import predict_tfidf
from src.intelligence.clustering import embed_reviews, cluster_reviews, build_topic_reviews
from src.intelligence.summarizer import generate_bullets
import joblib
from sklearn.pipeline import Pipeline
from src.config import RAW_REVIEWS_PATH, TFIDF_VECTORIZER_PATH, LR_MODEL_PATH

# load tfidf once
vectorizer     = joblib.load(TFIDF_VECTORIZER_PATH)
lr_model       = joblib.load(LR_MODEL_PATH)
tfidf_pipeline = Pipeline([('tfidf', vectorizer), ('lr', lr_model)])

# pick your demo products — find valid ASINs from your dataset first
cache = {}

for asin, name in demo_products_.items():
    print(f'Processing {name}...')
    reviews = stream_reviews_for_asin(str(RAW_REVIEWS_PATH), asin)

    if not reviews:
        print(f'  No reviews found, skipping.')
        continue

    clean   = [clean_text(r) for r in reviews]
    clean   = [r for r in clean if r]
    labels  = predict_tfidf(tfidf_pipeline, clean)

    total = len(labels)
    breakdown = {
        'positive': round(labels.count('positive') / total * 100, 1),
        'neutral':  round(labels.count('neutral')  / total * 100, 1),
        'negative': round(labels.count('negative') / total * 100, 1),
    }

    embeddings    = embed_reviews(reviews)
    topics, _     = cluster_reviews(reviews, embeddings)
    topic_reviews = build_topic_reviews(reviews, topics)
    praise, complaints = generate_bullets(topic_reviews)

    cache[asin] = {
        'name':       name,
        'total':      total,
        'breakdown':  breakdown,
        'praise':     praise,
        'complaints': complaints,
    }
    print(f'  Done — {total} reviews')

with open('data/processed/demo_cache.json', 'w') as f:
    json.dump(cache, f, indent=2)

print('Cache saved.')

In [ ]:
import gzip
import json
import pandas as pd
from src.config import RAW_REVIEWS_PATH, RAW_METADATA_PATH

# get all ASINs that have reviews in the dataset
print('Collecting ASINs from reviews file...')
review_asins = set()

with gzip.open(RAW_REVIEWS_PATH, 'rt', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        asin   = record.get('asin')
        if asin:
            review_asins.add(asin)

print(f'Unique ASINs with reviews: {len(review_asins):,}')

# extract title + asin from metadata for matching ASINs only
print('Extracting product names from metadata...')
products = []

with gzip.open(RAW_METADATA_PATH, 'rt', encoding='utf-8') as f:
    for line in f:
        try:
            record = json.loads(line)
            asin   = record.get('asin')
            title  = record.get('title', '').strip()
            if asin in review_asins and title:
                products.append({'asin': asin, 'title': title})
        except:
            continue

df = pd.DataFrame(products).drop_duplicates(subset='asin')
df.to_csv('data/processed/product_lookup.csv', index=False)

print(f'Saved {len(df):,} products to product_lookup.csv')
print(f'File size: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB')